![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Featureform On-Demand (Real-Time) Features

In this recipe we define an **on-demand feature** in [**Featureform**](https://docs.featureform.com/) — a feature computed **at request time**, from the live request payload combined with a precomputed feature served from **Redis**.

## Why on-demand features
Some features can't be precomputed because they depend on data that only exists *at the moment of the request* — the amount of the transaction being scored right now, the user's current cart, the time of day. On-demand features let you express that last-mile computation **as a versioned Featureform resource** instead of scattered application code, so the logic that ran in training is the exact logic that runs in production.

## What we'll build
A fraud-style **risk ratio**: `incoming transaction amount ÷ the user's historical average`.
- The **historical average** is a normal feature, precomputed by a SQL transformation and materialized to **Redis**.
- The **incoming amount** is passed in at request time as a parameter.
- An **on-demand feature** combines the two when you call `client.features(...)`.

## The stack — all local, no Spark

- **ClickHouse** — offline store; runs the SQL transformation for the historical average.
- **Redis** — online store; serves that average at low latency.
- **Featureform** coordinator.

> ⚠️ **Needs local Docker; will not run on Colab or in CI.** The setup cells below launch everything this recipe needs — the Featureform coordinator (gRPC `localhost:7878`, dashboard `http://localhost`), ClickHouse, and Redis — starting each only if it isn't already running. You just need a running Docker daemon.

## Environment Setup

### Install Python Dependencies

In [1]:
%pip install -q featureform redis clickhouse-connect numpy

Note: you may need to restart the kernel to use updated packages.


### Start Featureform, ClickHouse, and Redis

`featureform deploy docker` starts the coordinator and is a no-op if it's already running. ClickHouse and Redis are (re)started fresh.

In [2]:
# NBVAL_SKIP
# This notebook owns ClickHouse and Redis: start them fresh here and remove them
# in the cleanup cell. The Featureform coordinator may be shared with / managed by
# other tooling, so we only *ensure* it is up (deploy is a no-op if it already is)
# and never remove it.
#
# No ports are published to the host: the coordinator reaches these containers
# over the shared docker network by container IP, and the data-load cell talks to
# ClickHouse via `docker exec`. Publishing ports would collide with other host
# tools (e.g. Jupyter kernels grab ~9000, ClickHouse's native port).
!docker rm -f clickhouse redis 2>/dev/null
!docker run -d --name clickhouse -e CLICKHOUSE_SKIP_USER_SETUP=1 clickhouse/clickhouse-server:latest
!docker run -d --name redis redis:8

# Ensure the coordinator is running, then wait until it reports healthy — a
# coordinator that is up but not yet ready fails the ClickHouse transformation.
!featureform deploy docker
!for i in $(seq 1 40); do [ "$(docker inspect -f '{{.State.Health.Status}}' featureform 2>/dev/null)" = healthy ] && echo "coordinator healthy" && break; sleep 3; done

f6b00917c2e5cf1eb79317029d0d4b0a08eef1340a8a2dc5d09115b9ebfd40e6
e91cec91410eda1f91d826a938fc6620f8ebad6d669a7cf845e21d27a2f6c81f
Deploying Featureform on Docker
Starting Docker deployment on Darwin 24.6.0
Checking if featureform container exists...
	Container featureform has status "exited"
	Container featureform is stopped. Starting...

Featureform is now running!
To access the dashboard, visit http://localhost:80
To apply definition files, run `featureform apply <file.py> --host http://localhost:7878 --insecure`


### Configure connections

In [3]:
import os
import time
import uuid
import subprocess

# Featureform coordinator (gRPC)
FEATUREFORM_HOST = os.getenv("FEATUREFORM_HOST", "localhost:7878")


def _container_ip(name):
    """IP of a container on the shared docker bridge network."""
    return subprocess.run(
        ["docker", "inspect", "-f",
         "{{range .NetworkSettings.Networks}}{{.IPAddress}}{{end}}", name],
        capture_output=True, text=True,
    ).stdout.strip()


# The coordinator shares the default docker bridge with the provider containers,
# so it connects to them by container IP + internal port. Using the internal
# network (instead of host.docker.internal + published ports) avoids collisions
# with other host tools — notably Jupyter kernels, which grab ports around 9000,
# ClickHouse's native port. Override with env vars if your setup differs.
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST") or _container_ip("clickhouse")
CLICKHOUSE_NATIVE_PORT = int(os.getenv("CLICKHOUSE_NATIVE_PORT", "9000"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "default")

REDIS_HOST = os.getenv("REDIS_HOST") or _container_ip("redis")
REDIS_PORT = int(os.getenv("REDIS_PORT", "6379"))
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

print("clickhouse:", CLICKHOUSE_HOST, "redis:", REDIS_HOST)

# The coordinator's metadata is persistent (survives restarts), and Featureform
# dedups equivalent resources and reuses their variant — which, against state a
# previous run left behind, silently rewires a source to the wrong variant and
# breaks the graph. Registering fresh *names* each run makes every run a clean,
# self-contained apply that never collides with or reuses prior state.
SUFFIX = uuid.uuid4().hex[:8]
VARIANT = "quickstart"
TX_NAME = "transactions_" + SUFFIX
AVG_SRC_NAME = "average_user_transaction_" + SUFFIX
AVG_FEAT_NAME = "avg_transactions_" + SUFFIX
ONDEMAND_NAME = "transaction_risk_ratio_" + SUFFIX

# Other systems may restart the coordinator at any moment, and it needs a few
# seconds to warm up after launch. Wrap coordinator calls so transient gRPC
# failures are retried instead of aborting the notebook.
_TRANSIENT = ("could not connect", "socket closed", "unavailable", "connection refused")

def with_ff_retry(fn, attempts=40, delay=5):
    for _i in range(attempts):
        try:
            return fn()
        except Exception as _e:
            if _i < attempts - 1 and any(s in str(_e).lower() for s in _TRANSIENT):
                time.sleep(delay)
                continue
            raise

clickhouse: 172.17.0.2 redis: 172.17.0.3


### Create a sample transactions table in ClickHouse

We load a small transactions table so the average-transaction feature has data to aggregate.

In [4]:
# NBVAL_SKIP
import subprocess
import numpy as np

def ch(query, stdin=None):
    """Run a ClickHouse query inside the container (no host ports involved)."""
    return subprocess.run(
        ["docker", "exec", "-i", "clickhouse", "clickhouse-client", "--query", query],
        input=stdin, text=True, capture_output=True,
    )

# Wait for clickhouse-server inside the container to start accepting queries.
for _ in range(30):
    if ch("SELECT 1").returncode == 0:
        break
    time.sleep(2)

ch("DROP TABLE IF EXISTS transactions")
ch("CREATE TABLE transactions (TransactionID String, CustomerID String, "
   "TransactionAmount Float64) ENGINE = MergeTree ORDER BY CustomerID")

rng = np.random.default_rng(42)
csv = "".join(
    f"T{i:05d},C{int(rng.integers(1000, 1050)):04d},{round(float(rng.gamma(2.0, 50.0)), 2)}\n"
    for i in range(500)
)
ch("INSERT INTO transactions FORMAT CSV", stdin=csv)
print("rows:", ch("SELECT count() FROM transactions").stdout.strip())

rows: 500


## Register providers, source, and the precomputed feature

Standard setup: register ClickHouse + Redis, a SQL transformation for each user's average transaction, and a feature materialized to Redis. This is the value the on-demand feature will build on.

In [5]:
import featureform as ff

clickhouse = ff.register_clickhouse(
    name="clickhouse-quickstart",
    description="ClickHouse offline store with transaction history",
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_NATIVE_PORT,
    user=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

redis = ff.register_redis(
    name="redis-quickstart",
    description="Redis online (inference) store",
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=0,
)

In [6]:
transactions = clickhouse.register_table(
    name="transactions", variant="quickstart", table="transactions",
)

@clickhouse.sql_transformation(variant="quickstart")
def average_user_transaction():
    return (
        "SELECT CustomerID AS user_id, avg(TransactionAmount) AS avg_transaction_amt "
        "FROM {{transactions.quickstart}} GROUP BY CustomerID"
    )

@ff.entity
class User:
    avg_transactions = ff.Feature(
        average_user_transaction[["user_id", "avg_transaction_amt"]],
        variant="quickstart",
        type=ff.Float32,
        inference_store=redis,
    )

## Define the on-demand feature

An on-demand feature is a function decorated with `@ff.ondemand_feature`. Its signature is fixed: `(client, params, entities)`.
- `client` — lets the function look up other (precomputed) features, e.g. from Redis.
- `entities` — the entity keys passed at serving time.
- `params` — arbitrary request-time inputs you supply per call.

Here it fetches the user's stored average from Redis and divides the live amount by it. This function is registered and versioned like any feature — but it runs **client-side, at request time**.

In [7]:
@ff.ondemand_feature(variant="quickstart")
def transaction_risk_ratio(client, params, entities):
    """Live transaction amount relative to the user's historical average."""
    avg = client.features([("avg_transactions", "quickstart")], {"user": entities["user"]})[0]
    incoming_amount = params[0]
    if not avg:
        return 0.0
    return float(incoming_amount) / float(avg)

## Apply

`client.apply()` materializes the average into Redis and registers the on-demand feature definition.

In [8]:
# NBVAL_SKIP
client = ff.Client(host=FEATUREFORM_HOST, insecure=True)
with_ff_retry(lambda: client.apply(asynchronous=False, verbose=True))

UserWarning: install "ipywidgets" for Jupyter support

Applying Run: 2026-07-23t17-35-43
Creating User default_owner 
Creating Provider clickhouse-quickstart 
Creating Provider redis-quickstart 
Creating Source Variant transactions quickstart
Creating Source Variant average_user_transaction quickstart
Creating Entity user 
Creating Feature Variant avg_transactions quickstart
Creating Ondemand Feature transaction_risk_ratio quickstart



## Serve it — combine live input with the Redis-served average

Pass the entity and the request-time `params` to `client.features()`. The same call would run behind a live fraud model: a ratio well above 1 means this transaction is large relative to the user's norm.

In [9]:
# NBVAL_SKIP
def _serve():
    user_id = client.dataframe(average_user_transaction)["user_id"].iloc[0]
    stored_avg = client.features([("avg_transactions", "quickstart")], {"user": user_id})[0]
    for incoming_amount in [stored_avg, stored_avg * 5]:
        ratio = client.features(
            [transaction_risk_ratio],
            {"user": user_id},
            params=[incoming_amount],
        )
        print(f"user {user_id}: amount={incoming_amount:.2f}  avg={stored_avg:.2f}  risk_ratio={ratio}")

with_ff_retry(_serve)

No resources to apply
user C1047: amount=92.00  avg=92.00  risk_ratio=[1.0]
user C1047: amount=460.01  avg=92.00  risk_ratio=[5.0]


### Why this matters

The division logic lives in **one versioned resource**, not duplicated across a training script and a serving service. Train on `transaction_risk_ratio` and you score production traffic with byte-for-byte the same computation — no training-serving skew, even for the real-time part.

## Cleanup

Stop and remove the containers when you're done.

In [10]:
# NBVAL_SKIP
# Remove the ClickHouse and Redis containers this notebook started. The
# coordinator is left alone in case it is shared with other tooling; stop it
# yourself with `!featureform stop docker` if this notebook launched it.
!docker rm -f clickhouse redis

clickhouse
redis


I0000 00:00:1784853534.317493 3481186 chttp2_transport.cc:1353] ipv6:%5B::1%5D:7878: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0000 00:00:1784853534.317794 3481186 chttp2_transport.cc:1385] ipv6:%5B::1%5D:7878: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 60000ms


## Learn more

- [Featureform on-demand features](https://docs.featureform.com/)
- [Featureform + Redis fraud detection recipe](./02_featureform_fraud_detection.ipynb)